# Cardiac Fairness Pipeline - End-to-End Demo

Complete fairness-aware ML pipeline for cardiac disease prediction.

**Pipeline Stages:**
1. Data loading and profiling
2. Preprocessing and splitting
3. Baseline model training
4. Fairness assessment
5. Visualization

**Note**: For detailed experiments (age binning, mitigation), see notebooks in `experiments/`.

In [1]:
# Standard libraries
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data and viz
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Add project to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

# Import FairXAI modules
from fairxai.data.loaders import CardiacDataLoader
from fairxai.data.preprocessors import CardiacPreprocessor
from fairxai.data.profilers import DataProfiler
from fairxai.models.baseline import BaselineLogisticRegression
from fairxai.fairness.metrics import FairnessMetrics
from fairxai.visualization.plots import (
    plot_distribution,
    plot_correlation_matrix,
    plot_model_performance,
    plot_fairness_heatmap
)

print("✓ All imports successful")
print(f"Project root: {project_root}")

✓ All imports successful
Project root: /home/miguel/Desktop/Dissertacao/Code/FairXAI


## Stage 1: Data Loading

Load and profile both cardiac datasets.

In [ ]:
# Load raw standardized datasets directly
data_dir = project_root / 'data/raw/cardiac'

cleveland = pd.read_csv(data_dir / 'cleveland_standardized.csv')
kaggle_heart = pd.read_csv(data_dir / 'kaggle_heart_standardized.csv')

print("Dataset Overview:")
print(f"\nCleveland: {cleveland.shape}")
print(f"  Positive cases: {cleveland['heart_disease'].sum()} ({cleveland['heart_disease'].mean()*100:.1f}%)")
print(f"  Age range: {cleveland['age_raw'].min():.0f}-{cleveland['age_raw'].max():.0f}")

print(f"\nKaggle Heart: {kaggle_heart.shape}")
print(f"  Positive cases: {kaggle_heart['heart_disease'].sum()} ({kaggle_heart['heart_disease'].mean()*100:.1f}%)")
print(f"  Age range: {kaggle_heart['age_raw'].min():.0f}-{kaggle_heart['age_raw'].max():.0f}")

TypeError: CardiacDataLoader.__init__() missing 1 required positional argument: 'config_path'

## Stage 2: Data Profiling

Analyze distributions and check for data quality issues.

In [ ]:
# Profile Cleveland dataset
profiler = DataProfiler()
profile = profiler.profile_dataset(cleveland, dataset_name='Cleveland')

print("Cleveland Dataset Profile:")
print(f"\n  Numeric features: {profile['n_numeric']}")
print(f"  Categorical features: {profile['n_categorical']}")
print(f"  Missing values: {profile['total_missing']} ({profile['missing_percentage']:.1f}%)")
print(f"  Duplicates: {profile['n_duplicates']}")

# Show feature stats
print("\nFeature Statistics:")
for feature, stats in list(profile['numeric_stats'].items())[:5]:
    print(f"  {feature}: mean={stats['mean']:.2f}, std={stats['std']:.2f}")

## Stage 3: Preprocessing

Split data, scale features, and create sensitive attribute groups.

In [ ]:
# Load preprocessed data (already split and scaled)
processed_dir = project_root / 'data/processed/cardiac'
dataset_name = 'cleveland'

train_df = pd.read_csv(processed_dir / f'{dataset_name}_train.csv')
test_df = pd.read_csv(processed_dir / f'{dataset_name}_test.csv')

# Encode sex if categorical
if train_df['sex'].dtype == 'object':
    sex_map = {'Male': 1, 'Female': 0, 'M': 1, 'F': 0}
    train_df['sex'] = train_df['sex'].map(sex_map)
    test_df['sex'] = test_df['sex'].map(sex_map)
    print("✓ Encoded categorical 'sex' variable")

# Separate features, target, and sensitive attributes
# Exclude target, sensitive attrs, metadata, and original categorical columns
exclude_cols = [
    'heart_disease', 'age_group', 'sex', 'sex_extended', 'sex_bin',
    'Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope',
    '_dataset_source', '_dataset_file', 'age_raw', 'HeartDisease'
]
exclude_cols = [col for col in exclude_cols if col in train_df.columns]

X_train = train_df.drop(columns=exclude_cols)
y_train = train_df['heart_disease']
sensitive_train = train_df[['age_group', 'sex']]

X_test = test_df.drop(columns=exclude_cols)
y_test = test_df['heart_disease']
sensitive_test = test_df[['age_group', 'sex']]

print(f"✓ Loaded preprocessed data")
print(f"  Train set: {X_train.shape}")
print(f"  Test set: {X_test.shape}")
print(f"  Features: {list(X_train.columns)[:10]}...")

# Check sensitive attributes
print(f"\nSensitive Attributes:")
print(f"  Sex distribution: {sensitive_train['sex'].value_counts().to_dict()}")
print(f"  Age groups: {sensitive_train['age_group'].value_counts().sort_index().to_dict()}")

## Stage 4: Baseline Model Training

Train Logistic Regression baseline.

In [ ]:
# Train baseline model
baseline = BaselineLogisticRegression(random_state=42)
baseline.fit(X_train, y_train)

# Evaluate
metrics = baseline.evaluate(X_test, y_test)

print("Baseline Model Performance:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.3f}")

# Check clinical constraint
if metrics['recall'] >= 0.70:
    print(f"\n✓ Meets clinical constraint (recall ≥ 0.70)")
else:
    print(f"\n⚠️  Below clinical threshold (recall = {metrics['recall']:.3f})")

## Stage 5: Fairness Assessment

Evaluate fairness across sensitive attributes.

In [ ]:
# Generate predictions
y_pred = baseline.predict(X_test_clean)
y_proba = baseline.predict_proba(X_test_clean)[:, 1]

# Create predictions DataFrame
predictions_df = pd.DataFrame({
    'y_true': y_test.values,
    'y_pred': y_pred,
    'y_proba': y_proba,
    'sex': sensitive_test['sex'].values,
    'age_group': sensitive_test['age_group'].values
})

# Calculate fairness metrics
fairness_calc = FairnessMetrics()
fairness_results = fairness_calc.calculate_all_metrics(
    predictions_df,
    sensitive_attributes=['sex', 'age_group'],
    privileged_groups={'sex': 1, 'age_group': '60-69'}
)

print("Fairness Assessment:")
print("\n1. Demographic Parity (Sex):")
dp_sex = fairness_results['demographic_parity']['sex']
for group, rate in dp_sex['group_rates'].items():
    print(f"     Group {group}: {rate:.3f}")
print(f"   Max difference: {dp_sex['max_difference']:.3f}")

print("\n2. Demographic Parity (Age Group):")
dp_age = fairness_results['demographic_parity']['age_group']
print(f"   Max difference: {dp_age['max_difference']:.3f}")

print("\n3. Equalized Odds (Sex):")
eo_sex = fairness_results['equalized_odds']['sex']
print(f"   TPR difference: {eo_sex['tpr_difference']:.3f}")
print(f"   FPR difference: {eo_sex['fpr_difference']:.3f}")

## Stage 6: Visualization

Visualize model performance and fairness metrics.

In [ ]:
# Performance visualization
fig = plot_model_performance(
    y_test.values,
    y_pred,
    y_proba,
    model_name='Baseline Logistic Regression'
)
plt.tight_layout()
plt.show()

print("✓ Performance visualization complete")

In [ ]:
# Fairness heatmap
fig = plot_fairness_heatmap(
    fairness_results,
    title='Fairness Metrics: Cleveland Dataset'
)
plt.tight_layout()
plt.show()

print("✓ Fairness visualization complete")

## Summary

**Pipeline Results:**
- ✓ Data loaded and profiled
- ✓ Preprocessing complete (stratified split, scaling)
- ✓ Baseline model trained
- ✓ Clinical performance evaluated
- ✓ Fairness assessed across demographics

**Next Steps:**
1. **Age Binning**: Run `experiments/01_age_binning_strategies.ipynb` to optimize discretization
2. **Mitigation**: Run `experiments/02_mitigation_comparison.ipynb` to improve fairness
3. **Production**: Use `scripts/cardiac_pipeline.sh` for batch processing

**For detailed experiments, see:**
- `notebooks/experiments/01_age_binning_strategies.ipynb`
- `notebooks/experiments/02_mitigation_comparison.ipynb`